In [8]:
import os
from dotenv import load_dotenv

from youtube_transcript_api import YouTubeTranscriptApi
from langchain_core.documents import Document

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# -----------------------------------
# 1️⃣ Load Environment Variables
# -----------------------------------

load_dotenv()

# Make sure OPENAI_API_KEY is in your .env file
# OPENAI_API_KEY=your_key_here

# -----------------------------------
# 2️⃣ Fetch YouTube Transcript
# -----------------------------------

video_id = "hXb1k59w3M8"

print("Fetching transcript...")

ytt_api = YouTubeTranscriptApi()

transcript = ytt_api.fetch(video_id)

full_text = " ".join([t.text for t in transcript])

documents = [Document(page_content=full_text)]

print("Transcript loaded successfully.")

# -----------------------------------
# 3️⃣ Text Splitting
# -----------------------------------

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = text_splitter.split_documents(documents)

print(f"Split into {len(docs)} chunks.")

# -----------------------------------
# 4️⃣ Create Embeddings + Vector Store
# -----------------------------------

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    docs,
    embedding=embeddings,
    persist_directory="./youtube_rag_db"
)

retriever = vectorstore.as_retriever()

print("Vector store created.")

# -----------------------------------
# 5️⃣ Create Prompt
# -----------------------------------

prompt = ChatPromptTemplate.from_template("""
You are an assistant answering questions about a YouTube video.

Use ONLY the context below to answer the question.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question:
{question}

Answer:
""")

# -----------------------------------
# 6️⃣ Initialize LLM
# -----------------------------------

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# -----------------------------------
# 7️⃣ Build Runnable RAG Chain
# -----------------------------------

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG system ready.")

# -----------------------------------
# 8️⃣ Ask Questions
# -----------------------------------

while True:
    query = input("\nAsk about the video (type 'exit' to quit): ")

    if query.lower() == "exit":
        break

    response = rag_chain.invoke(query)
    print("\nAnswer:\n")
    print(response)

Fetching transcript...
Transcript loaded successfully.
Split into 32 chunks.
Vector store created.
RAG system ready.

Answer:

The agenda of the meeting includes having conversations about technology, specifically focusing on AI, robotics, energy, and space. The goal is to discuss the meaningful components of these technologies, their engineering challenges, and how they can maximize the future of civilization and expand consciousness beyond Earth. Additionally, there is an emphasis on understanding and resolving various issues through dialogue.

Answer:

I don't know.

Answer:

This discussion is happening in Davos.

Answer:

1. AI Progression: The speaker predicts that AI may surpass human intelligence by the end of this year or next year, and by 2030 or 2031, AI could be smarter than all of humanity collectively.

2. Inspiration and Curiosity: The speaker reflects on their childhood interest in science fiction as a foundation for their curiosity and innovation.

3. Investment Potent